In [1]:
from pathlib import Path
DIR_PLANET = Path('~').expanduser() / 'Library/CloudStorage/Dropbox' / 'planet'

In [7]:
import geopandas as gpd
import pandas as pd
import json

# Load Site Data CSVs


In [3]:
phenocam_df = pd.read_csv(DIR_PLANET / 'qgis/sites/phenocam_sites.csv')
phenocam_df.rename(columns={c: f'(phenocam){c}' for c in list(phenocam_df.columns) if c not in ['site_name']}, inplace=True)
# phenocam_df

In [4]:
flux_df = pd.read_csv(DIR_PLANET / 'qgis/sites/AmeriFlux-site-search-results-202605032116.csv')
flux_df.rename(columns={c: f'(flux){c}' for c in list(flux_df.columns) if c not in ['Site ID']}, inplace=True)
# flux_df

In [5]:
study_sites_df = pd.read_csv(DIR_PLANET / 'qgis/sites/study_sites.csv')
study_sites_df[['latitude', 'longitude']] = study_sites_df['site_marker'].str.split(',', expand=True)
study_sites_df['latitude'] = study_sites_df['latitude'].astype(float)
study_sites_df['longitude'] = study_sites_df['longitude'].astype(float)
# study_sites_df

In [6]:
data_gdf = gpd.GeoDataFrame(study_sites_df, geometry=gpd.points_from_xy(study_sites_df.latitude, study_sites_df.longitude))
data_gdf = pd.merge(data_gdf, flux_df, on=['Site ID'], how='left')
data_gdf = pd.merge(data_gdf, phenocam_df, on=['site_name'], how='left')
data_gdf.rename(columns={'site_name': '(phenocam)site_name'}, inplace=True)
data_gdf.drop(columns=['(flux)Latitude (degrees)', '(flux)Longitude (degrees)'], axis=1, inplace=True)

# data_gdf

In [8]:
def split_lat_lon_string(js):
    data = json.loads(js)
    return list(zip(data['lat'], data['lng']))

data_gdf['site_polygon'] = data_gdf['site_polygon'].apply(split_lat_lon_string)

# Visualization of Sites


In [9]:
from ipyleaflet import AwesomeIcon, Map, Marker, Polygon, ScaleControl, basemaps
from ipywidgets import HTML, Layout

site_icon = AwesomeIcon(name='tower-cell', marker_color='blue', icon_color='blue', spin=False)
# phenocam_icon = AwesomeIcon(name='camera', marker_color='green', icon_color='green', spin=False)

center = (44, -103)
m = Map(center=center, zoom=5, basemap=basemaps.Esri.WorldImagery, layout=Layout(height='500px'))
m.add(ScaleControl(position='bottomleft'))


site_boundary_locations = []

for index, site in data_gdf.iterrows():
    # Build hover tooltip HTML
    flux_name = site.get('(flux)Name', 'N/A')
    site_id = site.get('Site ID', 'N/A')
    veg = site.get('(flux)Vegetation Abbreviation (IGBP)', 'N/A')
    base_start = site.get('(flux)AmeriFlux BASE Data Start', 'N/A')
    base_end = site.get('(flux)AmeriFlux BASE Data End', 'N/A')
    flux_start = site.get('(flux)AmeriFlux FLUXNET Data Start', 'N/A')
    flux_end = site.get('(flux)AmeriFlux FLUXNET Data End', 'N/A')
    pheno_name = site.get('(phenocam)site_name', 'N/A')
    pheno_start = site.get('(phenocam)date_first', 'N/A')
    pheno_end = site.get('(phenocam)date_last', 'N/A')

    tooltip_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; min-width: 180px;">
        <b style="font-size: 13px;">Flux: {flux_name} ({site_id})</b><hr style="margin: 4px 0;">
        <b>Vegetation:</b> {veg}<br>
        <b>Base data:</b> {base_start} - {base_end}<br>
        <b>Flux data:</b> {flux_start} - {flux_end}<br>
        <hr style="margin: 4px 0;">
        <b>PhenoCam:</b> {pheno_name}<br>
        <b>PhenoCam data:</b> {pheno_start} - {pheno_end}
    </div>
    """

    marker = Marker(
        icon=site_icon,
        location=(site['latitude'], site['longitude']),
        draggable=True,
        title=f"{flux_name} ({site_id})",
    )

    # Attach popup that opens on hover
    popup = HTML(value=tooltip_html)
    marker.popup = popup

    m.add(marker)
    site_boundary_locations.append(site['site_polygon'])

    # m.add(Marker(icon=phenocam_icon, location=(site['(phenocam)latitude'], site['(phenocam)longitude']), draggable=True, title=f'{site.get("(phenocam)site_name", "N/A")}'))

# Add polygons
site_boundary_polygons = Polygon(locations=site_boundary_locations, color='blue', fill_color='blue', fill_opacity=0.1, weight=2)
m.add(site_boundary_polygons)

display(m)

m.save('sites.html', title='My Map')

Map(center=[44, -103], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

# Create Selected Sites and Info CSV


In [10]:
data_gdf.rename(columns={'(flux)Name': 'name'}, inplace=True)

In [11]:
def underscore_name(row):
    return '_'.join(row.split(' '))

In [ ]:
data_gdf['site_name'] = data_gdf['name'].apply(underscore_name)

In [14]:
data_gdf.head(1)

,Flux Site Name,Site ID,(phenocam)site_name,site_url,site_marker,site_polygon,latitude,longitude,geometry,name,...,(flux)BASE variables available,(flux)FLUXNET variables available,(phenocam)site_url,(phenocam)latitude,(phenocam)longitude,(phenocam)elevation_m,(phenocam)active,(phenocam)date_first,(phenocam)date_last,site
0,Walnut Gulch Kendall Grasslands (WKG),US-Wkg,kendall,https://ameriflux.lbl.gov/sites/siteinfo/US-Wkg,"31.7365,-109.9419","[(31.7814910017996, -109.994712510874), (31.78...",31.7365,-109.9419,POINT (31.736 -109.942),Walnut Gulch Kendall Grasslands,...,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,https://phenocam.nau.edu/webcam/sites/kendall/,31.73652,-109.94185,1529.0,True,2012-07-06,2026-03-23,Walnut_Gulch_Kendall_Grasslands


In [ ]:
data_gdf[]

In [ ]:
data_gdf[['Site ID', 'site_name', 'site_marker', 'site_polygon']].to_csv('sites_info.csv')